In [ ]:
# 0. Verify kernel — path should end with VectorDbDemo/venv/bin/python
import sys
print(sys.executable)
print("Kernel OK")

# Complete Vector Database Tutorial

> **Kernel:** Use **Vector DB Demo** (`Kernel` → `Change kernel`).
> Packages live in `VectorDbDemo/venv/`, not `Services/.venv`.

In [ ]:
# 1. Install dependencies (skip if you already ran `make setup`)
!{sys.executable} -m pip -q install -r requirements.txt

In [ ]:
# 2. Verify chromadb is installed
!pip show chromadb

In [ ]:
# Download sample TechCrunch articles (uncomment if needed)
# !wget -q https://www.dropbox.com/s/vs6ocyvpzzncvwh/new_articles.zip

## Download data (optional)

Skip this section if you already have the `Articles/` folder (21 `.txt` files).

In [ ]:
# Unzip downloaded articles (uncomment after wget)
# !unzip -q new_articles.zip -d new_articles

## 3. Project setup

Loads `.env` from `VectorDbDemo/` and imports shared helpers from `chroma_utils.py`.
Run this once after restarting the kernel.

In [ ]:
import importlib
import os
import shutil
import sys
from pathlib import Path

for candidate in (Path.cwd(), Path.cwd() / "VectorDbDemo"):
    if (candidate / "chroma_utils.py").exists():
        sys.path.insert(0, str(candidate))
        break
else:
    raise ImportError("chroma_utils.py not found — open the notebook from VectorDbDemo/.")

import chroma_utils  # noqa: E402

importlib.reload(chroma_utils)
chroma_utils.load_project_env()

## 4. Imports

LangChain components for loading documents, embeddings, vector store, and the QA chain.

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

## 5. Load documents

Read all article `.txt` files from `./Articles/`.

In [ ]:
loader = DirectoryLoader("./Articles/", glob="./*.txt", loader_cls=TextLoader)
document = loader.load()
print(f"Loaded {len(document)} articles")

In [ ]:
# Preview loaded documents (page_content + source file metadata)
document

## 6. Split into chunks

Split long articles into ~1000-character chunks with 200-character overlap.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
text = text_splitter.split_documents(document)
print(f"Created {len(text)} chunks")
text[1]  # preview one chunk

## 7. Open or create Chroma database

**Single entry point for all persistence** — re-run this cell after a kernel restart.

| Condition | Action |
|-----------|--------|
| `chroma-local-db/` exists | Open existing database (no re-embedding) |
| Folder missing, `chroma-local-db.zip` exists | Restore from zip, then open |
| Neither exists | Embed documents and create a fresh database |

In [ ]:
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

(
    vectordb,
    persist_directory,
    collection_name,
    client_settings,
) = chroma_utils.open_or_create_vectordb(documents=text, embedding=embedding)

## 8. Similarity search (retriever)

Query the vector store and return the most relevant document chunks.

In [ ]:
retriever = vectordb.as_retriever()
docs = retriever.invoke("How much money did Microsoft raise?")
print(f"Retrieved {len(docs)} chunks")
docs

In [ ]:
# Tune how many chunks are returned (k=2)
retriever = vectordb.as_retriever(search_kwargs={"k": 2})
retriever.search_type, retriever.search_kwargs

## 9. Q&A chain

Combine retrieval with a local Hugging Face LLM to answer questions using retrieved context.


In [ ]:
# Load local TinyLlama LLM (uses HF_TOKEN from .env)
if not os.getenv("HF_TOKEN"):
    raise ValueError("HF_TOKEN not found. Add it to VectorDbDemo/.env and re-run section 3.")

llm = chroma_utils.get_hf_llm()

In [ ]:
llm

In [ ]:
qa_prompt = PromptTemplate.from_template(
    """Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

{context}

Question: {input}
Helpful Answer:"""
)

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
qa_chain = create_retrieval_chain(retriever, question_answer_chain)

In [ ]:
def process_llm_response(response):
    print(response["answer"])
    print("\nSources:")
    for doc in response["context"]:
        print(f"  - {doc.metadata.get('source', 'unknown')}")

In [ ]:
query = "How much money did Microsoft raise?"
llm_response = qa_chain.invoke({"input": query})
process_llm_response(llm_response)

In [ ]:
query = "What is the news about Pando?"
llm_response = qa_chain.invoke({"input": query})
process_llm_response(llm_response)

## 10. Backup and cleanup

| Action | What it does |
|--------|--------------|
| **Archive** | Zip `chroma-local-db/` to `chroma-local-db.zip` |
| **Delete** | Remove the on-disk database folder |
| **Restore** | Re-run **section 7** — it auto-restores from zip if the folder is missing |

In [ ]:
archive_path = chroma_utils.archive_persist_directory()
print(f"Created {archive_path}")

In [ ]:
if vectordb is not None:
    vectordb.delete_collection()
vectordb = None
removed = chroma_utils.reset_persist_directory()
print(f"Removed {removed}")
print("Re-run section 7 to restore from zip or create a fresh database.")